# 解析結果を CoNLL-U 形式で保存・読み込みする

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/16_parsing/conllu.ipynb)

NLP クックブック 16-13 ／ 構文解析

依存構造解析の結果を、標準的な CoNLL-U 形式でファイルに保存したり、UD のツリーバンクを読み込んだりしたい。

In [1]:
!pip install -q ginza ja_ginza conllu

In [2]:
import spacy

nlp = spacy.load("ja_ginza")

def to_conllu(doc):
    lines = []
    for sid, sent in enumerate(doc.sents, 1):
        lines += [f"# sent_id = {sid}", f"# text = {sent.text}"]
        for i, t in enumerate(sent, 1):
            head = 0 if t.dep_ == "ROOT" else t.head.i - sent.start + 1
            feats = str(t.morph) or "_"
            misc = "_" if t.whitespace_ else "SpaceAfter=No"
            lines.append("\t".join([str(i), t.text, t.lemma_, t.pos_, t.tag_, feats, str(head), t.dep_.lower() if t.dep_ == "ROOT" else t.dep_, "_", misc]))
        lines.append("")
    return "\n".join(lines) + "\n"

text = to_conllu(nlp("太郎は本を読んだ。花子は寝ていた。"))
with open("parsed.conllu", "w", encoding="utf-8") as f:
    f.write(text)
print(text)

# sent_id = 1
# text = 太郎は本を読んだ。
1	太郎	太郎	PROPN	名詞-固有名詞-人名-名	Reading=タロウ	5	nsubj	_	SpaceAfter=No
2	は	は	ADP	助詞-係助詞	Reading=ハ	1	case	_	SpaceAfter=No
3	本	本	NOUN	名詞-普通名詞-一般	Reading=ホン	5	obj	_	SpaceAfter=No
4	を	を	ADP	助詞-格助詞	Reading=ヲ	3	case	_	SpaceAfter=No
5	読ん	読む	VERB	動詞-一般	Inflection=五段-マ行;連用形-撥音便|Reading=ヨン	0	root	_	SpaceAfter=No
6	だ	だ	AUX	助動詞	Inflection=助動詞-タ;終止形-一般|Reading=ダ	5	aux	_	SpaceAfter=No
7	。	。	PUNCT	補助記号-句点	Reading=。	5	punct	_	SpaceAfter=No

# sent_id = 2
# text = 花子は寝ていた。
1	花子	花子	PROPN	名詞-固有名詞-人名-名	Reading=ハナコ	3	nsubj	_	SpaceAfter=No
2	は	は	ADP	助詞-係助詞	Reading=ハ	1	case	_	SpaceAfter=No
3	寝	寝る	VERB	動詞-一般	Inflection=下一段-ナ行;連用形-一般|Reading=ネ	0	root	_	SpaceAfter=No
4	て	て	SCONJ	助詞-接続助詞	Reading=テ	3	mark	_	SpaceAfter=No
5	い	いる	VERB	動詞-非自立可能	Inflection=上一段-ア行;連用形-一般|Reading=イ	4	fixed	_	SpaceAfter=No
6	た	た	AUX	助動詞	Inflection=助動詞-タ;終止形-一般|Reading=タ	3	aux	_	SpaceAfter=No
7	。	。	PUNCT	補助記号-句点	Reading=。	3	punct	_	SpaceAfter=No




In [3]:
from conllu import parse

with open("parsed.conllu", encoding="utf-8") as f:
    sentences = parse(f.read())
for sent in sentences:
    print(sent.metadata["text"], [(t["form"], t["upos"], t["head"], t["deprel"]) for t in sent])
print(sentences[0].to_tree().token["form"], "が ROOT")

太郎は本を読んだ。 [('太郎', 'PROPN', 5, 'nsubj'), ('は', 'ADP', 1, 'case'), ('本', 'NOUN', 5, 'obj'), ('を', 'ADP', 3, 'case'), ('読ん', 'VERB', 0, 'root'), ('だ', 'AUX', 5, 'aux'), ('。', 'PUNCT', 5, 'punct')]
花子は寝ていた。 [('花子', 'PROPN', 3, 'nsubj'), ('は', 'ADP', 1, 'case'), ('寝', 'VERB', 0, 'root'), ('て', 'SCONJ', 3, 'mark'), ('い', 'VERB', 4, 'fixed'), ('た', 'AUX', 3, 'aux'), ('。', 'PUNCT', 3, 'punct')]
読ん が ROOT


- 10 列: ID, FORM, LEMMA, UPOS, XPOS, FEATS, HEAD, DEPREL, DEPS, MISC。列はタブ区切り、文は空行区切り。
- GiNZA はコマンドラインでも `ginza -f conllu < input.txt` で CoNLL-U を出力できる。